## Merge LMP Yearly Averages with Pnode Locations

Merges `va_lmp_yearly_avg_all.csv` with `va_pnode_ids_final_full.csv` to add latitude and longitude to each pnode, then adds census tract via spatial join.

In [5]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

DATA_DIR = next(
    p / "data" for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data").exists()
)

usa_dir = DATA_DIR / "processed" / "usa_data"

In [6]:
# Load data
lmp = pd.read_csv(usa_dir / "va_lmp_yearly_avg_all.csv")
pnodes = pd.read_csv(usa_dir / "va_pnode_ids_final_full.csv")

print("LMP shape:", lmp.shape)
print("Pnode shape:", pnodes.shape)
print("\nPnode columns:", pnodes.columns.tolist())

LMP shape: (8413, 9)
Pnode shape: (1152, 6)

Pnode columns: ['substation_name', 'latitude', 'longitude', 'pnode_match', 'pnode_id', 'type']


In [7]:
# Merge on pnode_id, adding only latitude and longitude
merged = lmp.merge(
    pnodes[["pnode_id", "latitude", "longitude"]],
    on="pnode_id",
    how="left"
)

print("Merged shape:", merged.shape)
print("Rows missing lat/lon:", merged["latitude"].isna().sum())
merged.head()

Merged shape: (8413, 11)
Rows missing lat/lon: 0


,year,pnode_id,pnode_name,type,avg_total_lmp,avg_energy,avg_congestion,avg_loss,hours,latitude,longitude
0,2018,48907,GOSHEN,LOAD,34.209886,35.695331,-2.101719,0.616194,8759,37.968345,-79.509225
1,2018,48908,GOSHEN,LOAD,34.208825,35.695331,-2.101741,0.615208,8759,37.968345,-79.509225
2,2018,49283,ROCKRIDG,LOAD,39.168442,35.695331,2.783761,0.689494,8759,37.798442,-79.415567
3,2018,49284,ROCKRIDG,LOAD,39.160176,35.695331,2.783762,0.681192,8759,37.798442,-79.415567
4,2018,49285,ROCKRIDG,LOAD,38.625711,35.695331,2.329300,0.601242,8759,37.798442,-79.415567


In [8]:
# Load Virginia census tracts
census = gpd.read_file(
    DATA_DIR / "raw" / "usa_data" / "virginia_census_data" / "cb_2024_51_tract_500k" / "cb_2024_51_tract_500k.shp"
)
print("Census CRS:", census.crs)
print("Census columns:", census.columns.tolist())

Census CRS: EPSG:4269
Census columns: ['STATEFP', 'COUNTYFP', 'TRACTCE', 'GEOIDFQ', 'GEOID', 'NAME', 'NAMELSAD', 'STUSPS', 'NAMELSADCO', 'STATE_NAME', 'LSAD', 'ALAND', 'AWATER', 'geometry']


In [9]:
# Convert merged data to GeoDataFrame using lat/lon
merged_geo = gpd.GeoDataFrame(
    merged,
    geometry=gpd.points_from_xy(merged["longitude"], merged["latitude"]),
    crs="EPSG:4326"
)

# Reproject census to match pnode CRS
census = census.to_crs("EPSG:4326")

# Spatial join — find which census tract each pnode falls in
merged_with_tract = gpd.sjoin(
    merged_geo,
    census[["GEOID", "NAME", "geometry"]],
    how="left",
    predicate="within"
)

# Drop geometry columns
merged_with_tract = merged_with_tract.drop(columns=["geometry", "index_right"])

print(f"Rows: {len(merged_with_tract):,}")
print(f"Rows missing census tract: {merged_with_tract['GEOID'].isna().sum()}")
merged_with_tract.head()

Rows: 8,413
Rows missing census tract: 0


,year,pnode_id,pnode_name,type,avg_total_lmp,avg_energy,avg_congestion,avg_loss,hours,latitude,longitude,GEOID,NAME
0,2018,48907,GOSHEN,LOAD,34.209886,35.695331,-2.101719,0.616194,8759,37.968345,-79.509225,51163930200,9302
1,2018,48908,GOSHEN,LOAD,34.208825,35.695331,-2.101741,0.615208,8759,37.968345,-79.509225,51163930200,9302
2,2018,49283,ROCKRIDG,LOAD,39.168442,35.695331,2.783761,0.689494,8759,37.798442,-79.415567,51163930101,9301.01
3,2018,49284,ROCKRIDG,LOAD,39.160176,35.695331,2.783762,0.681192,8759,37.798442,-79.415567,51163930101,9301.01
4,2018,49285,ROCKRIDG,LOAD,38.625711,35.695331,2.329300,0.601242,8759,37.798442,-79.415567,51163930101,9301.01


In [10]:
# Save final output
merged_with_tract.to_csv(usa_dir / "va_lmp_yearly_avg_geo.csv", index=False)
print(f"Saved {len(merged_with_tract):,} rows to va_lmp_yearly_avg_geo.csv")

Saved 8,413 rows to va_lmp_yearly_avg_geo.csv
